## *1. Problem Overview*

Building energy efficiency modeling is a critical domain in computational sustainability and urban energy management. Commercial and residential facilities contribute significantly to regional greenhouse gas emissions and electrical grid loads. Accurately modeling Site Energy Use Intensity (`site_eui`), expressed in kBtu/sq.ft, allows municipal planners, building managers, and policy makers to benchmark energy performance, identify energy-inefficient facilities, and prioritize retrofitting projects across diverse property types.


### *Problem Statement*

Given a comprehensive dataset of commercial and residential building characteristics, climate variables, and historical temperature metrics across various states, the challenge is to formulate a robust predictive regression framework. The model must accurately estimate `site_eui` based on facility attributes (such as `floor_area`, `year_built`, `energy_star_rating`, and `facility_type`) and local climatic indicators (such as heating and cooling degree days), despite high right-skewness, missing data, and potential linear dependency among predictors.


### *Regression Objective*

The objective of this project is to construct an end-to-end regression modeling pipeline to predict building Site Energy Use Intensity (`site_eui`). The pipeline covers raw dataset auditing, missing value treatment, duplicate removal, exploratory data analysis (EDA), domain feature engineering, zero-leakage preprocessing, and training, hyperparameter tuning, and comparative evaluation across all 10 required regression algorithms (Linear Regression, Ridge Regression, Lasso Regression, ElasticNet Regression, Polynomial Regression, Decision Tree Regressor, Random Forest Regressor, Gradient Boosting Regressor, Support Vector Regression, and K-Nearest Neighbors). All models are trained and evaluated on a unified stratified 80:20 train-test split using $R^2$, Root Mean Squared Error (RMSE), and Mean Absolute Error (MAE) metrics, with 5-fold cross-validation applied to the top-performing models, culminating in a single consolidated model comparison table as required by the Review 1 Capstone rubric.


## *2. Import Required Libraries*

We import core Python data manipulation libraries (`pandas`, `numpy`), visualization frameworks (`matplotlib.pyplot`, `seaborn`), and scikit-learn modules for train-test splitting, feature preprocessing, baseline linear models, grid search tuning, regression metrics, and model persistence via `joblib`.

- **Data Manipulation:** `pandas` for DataFrame manipulation, `numpy` for log/exp transformations and vector math.
- **Visualization:** `matplotlib.pyplot` and `seaborn` (configured with a **colorblind-friendly color palette**) for distribution plots, correlation heatmaps, scatter plots, and diagnostic residual plots.
- **Preprocessing & Persistence:** `StandardScaler`, `OneHotEncoder`, `PolynomialFeatures`, `SimpleImputer`, `ColumnTransformer`, and `joblib` to assemble and export leakage-free pipeline objects.
- **Regression Algorithms:** `LinearRegression`, `Ridge`, `Lasso`, and `ElasticNet` for baseline linear algorithms.
- **Model Evaluation:** `r2_score`, `mean_squared_error`, `mean_absolute_error` for rigorous performance evaluation.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from IPython.display import display

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Set global formatting and colorblind-friendly plotting styles (Rubric A2)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
print("Libraries successfully imported and global colorblind-friendly style initialized.")


Libraries successfully imported and global colorblind-friendly style initialized.


## *3. Dataset Loading*


We load the raw building dataset (`train.csv`) into a pandas DataFrame.


In [2]:
df_raw = pd.read_csv('train.csv')
print("Raw dataset loaded successfully.")


Raw dataset loaded successfully.


We check the dataset dimensions (rows and columns) in an isolated cell (Rubric A1).


In [3]:
print("Dataset Shape:", df_raw.shape)


Dataset Shape: (75757, 64)


We inspect column data types in an isolated cell with a clean tabular summary (Rubric A1).


In [4]:
df_dtypes_summary = pd.DataFrame({
    'Data Type': df_raw.dtypes.value_counts().index.astype(str),
    'Column Count': df_raw.dtypes.value_counts().values
})
print("Column Data Types Summary:")
display(df_dtypes_summary)


Column Data Types Summary:


,Data Type,Column Count
0,int64,37
1,float64,24
2,object,3


We audit missing-value counts per column in an isolated cell (Rubric A1).


In [5]:
missing_counts_audit = df_raw.isnull().sum()
missing_counts_audit = missing_counts_audit[missing_counts_audit > 0].sort_values(ascending=False)
print("=== MISSING VALUE COUNTS PER COLUMN ===")
print(missing_counts_audit)


=== MISSING VALUE COUNTS PER COLUMN ===
days_with_fog                45796
direction_peak_wind_speed    41811
direction_max_wind_speed     41082
max_wind_speed               41082
energy_star_rating           26709
year_built                    1837
dtype: int64


We compute summary distribution statistics for the target variable `site_eui` in an isolated cell (Rubric A1).


In [6]:
df_target_stats = pd.DataFrame(df_raw['site_eui'].describe()).reset_index()
df_target_stats.columns = ['Statistic', 'site_eui (kBtu/sq.ft)']
df_target_stats['site_eui (kBtu/sq.ft)'] = df_target_stats['site_eui (kBtu/sq.ft)'].round(2)

print("=== TARGET VARIABLE ('site_eui') DISTRIBUTION SUMMARY STATS ===")
display(df_target_stats)


=== TARGET VARIABLE ('site_eui') DISTRIBUTION SUMMARY STATS ===


,Statistic,site_eui (kBtu/sq.ft)
0,count,75757.00
1,mean,82.58
2,std,58.26
3,min,1.00
4,25%,54.53
5,50%,75.29
6,75%,97.28
7,max,997.87


We display the first five records of the dataset.


In [7]:
df_raw.head()


,Year_Factor,State_Factor,building_class,facility_type,floor_area,year_built,energy_star_rating,ELEVATION,january_min_temp,january_avg_temp,...,days_above_80F,days_above_90F,days_above_100F,days_above_110F,direction_max_wind_speed,direction_peak_wind_speed,max_wind_speed,days_with_fog,site_eui,id
0,1,State_1,Commercial,Grocery_store_or_food_market,61242.0,1942.0,11.0,2.4,36,50.5,...,14,0,0,0,1.0,1.0,1.0,NaN,248.682615,0
1,1,State_1,Commercial,Warehouse_Distribution_or_Shipping_center,274000.0,1955.0,45.0,1.8,36,50.5,...,14,0,0,0,1.0,NaN,1.0,12.0,26.500150,1
2,1,State_1,Commercial,Retail_Enclosed_mall,280025.0,1951.0,97.0,1.8,36,50.5,...,14,0,0,0,1.0,NaN,1.0,12.0,24.693619,2
3,1,State_1,Commercial,Education_Other_classroom,55325.0,1980.0,46.0,1.8,36,50.5,...,14,0,0,0,1.0,NaN,1.0,12.0,48.406926,3
4,1,State_1,Commercial,Warehouse_Nonrefrigerated,66000.0,1985.0,100.0,2.4,36,50.5,...,14,0,0,0,1.0,1.0,1.0,NaN,3.899395,4


We display the last five records of the dataset.


In [8]:
df_raw.tail()


,Year_Factor,State_Factor,building_class,facility_type,floor_area,year_built,energy_star_rating,ELEVATION,january_min_temp,january_avg_temp,...,days_above_80F,days_above_90F,days_above_100F,days_above_110F,direction_max_wind_speed,direction_peak_wind_speed,max_wind_speed,days_with_fog,site_eui,id
75752,6,State_11,Commercial,Office_Uncategorized,20410.0,1995.0,8.0,36.6,28,43.451613,...,25,3,0,0,NaN,NaN,NaN,NaN,132.918411,75752
75753,6,State_11,Residential,5plus_Unit_Building,40489.0,1910.0,98.0,36.6,28,43.451613,...,25,3,0,0,NaN,NaN,NaN,NaN,39.483672,75753
75754,6,State_11,Commercial,Commercial_Other,28072.0,1917.0,NaN,36.6,26,36.612903,...,6,0,0,0,NaN,NaN,NaN,NaN,48.404398,75754
75755,6,State_11,Commercial,Commercial_Other,53575.0,2012.0,NaN,36.6,26,36.612903,...,6,0,0,0,NaN,NaN,NaN,NaN,592.022750,75755
75756,6,State_11,Residential,2to4_Unit_Building,23888.0,1974.0,51.0,36.6,27,36.935484,...,16,0,0,0,NaN,NaN,NaN,NaN,29.154684,75756


We print the complete list of column names in the raw dataset.


In [9]:
print("Dataset Column Names:\n", df_raw.columns.tolist())


Dataset Column Names:
 ['Year_Factor', 'State_Factor', 'building_class', 'facility_type', 'floor_area', 'year_built', 'energy_star_rating', 'ELEVATION', 'january_min_temp', 'january_avg_temp', 'january_max_temp', 'february_min_temp', 'february_avg_temp', 'february_max_temp', 'march_min_temp', 'march_avg_temp', 'march_max_temp', 'april_min_temp', 'april_avg_temp', 'april_max_temp', 'may_min_temp', 'may_avg_temp', 'may_max_temp', 'june_min_temp', 'june_avg_temp', 'june_max_temp', 'july_min_temp', 'july_avg_temp', 'july_max_temp', 'august_min_temp', 'august_avg_temp', 'august_max_temp', 'september_min_temp', 'september_avg_temp', 'september_max_temp', 'october_min_temp', 'october_avg_temp', 'october_max_temp', 'november_min_temp', 'november_avg_temp', 'november_max_temp', 'december_min_temp', 'december_avg_temp', 'december_max_temp', 'cooling_degree_days', 'heating_degree_days', 'precipitation_inches', 'snowfall_inches', 'snowdepth_inches', 'avg_temp', 'days_below_30F', 'days_below_20F', 

## *4. Dataset Audit*


We inspect non-null counts, column indices, and memory utilization using dataset info.


In [10]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75757 entries, 0 to 75756
Data columns (total 64 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Year_Factor                75757 non-null  int64  
 1   State_Factor               75757 non-null  object 
 2   building_class             75757 non-null  object 
 3   facility_type              75757 non-null  object 
 4   floor_area                 75757 non-null  float64
 5   year_built                 73920 non-null  float64
 6   energy_star_rating         49048 non-null  float64
 7   ELEVATION                  75757 non-null  float64
 8   january_min_temp           75757 non-null  int64  
 9   january_avg_temp           75757 non-null  float64
 10  january_max_temp           75757 non-null  int64  
 11  february_min_temp          75757 non-null  int64  
 12  february_avg_temp          75757 non-null  float64
 13  february_max_temp          75757 non-null  int

We generate numerical descriptive statistics for all numeric features.


In [11]:
df_raw.describe()


,Year_Factor,floor_area,year_built,energy_star_rating,ELEVATION,january_min_temp,january_avg_temp,january_max_temp,february_min_temp,february_avg_temp,...,days_above_80F,days_above_90F,days_above_100F,days_above_110F,direction_max_wind_speed,direction_peak_wind_speed,max_wind_speed,days_with_fog,site_eui,id
count,75757.000000,7.575700e+04,73920.000000,49048.000000,75757.000000,75757.000000,75757.000000,75757.000000,75757.000000,75757.000000,...,75757.000000,75757.000000,75757.000000,75757.000000,34675.000000,33946.000000,34675.000000,29961.000000,75757.000000,75757.000000
mean,4.367755,1.659839e+05,1952.306764,61.048605,39.506323,11.432343,34.310468,59.054952,11.720567,35.526837,...,82.709809,14.058701,0.279539,0.002442,66.552675,62.779974,4.190601,109.142051,82.584693,37878.000000
std,1.471441,2.468758e+05,37.053619,28.663683,60.656596,9.381027,6.996108,5.355458,12.577272,8.866697,...,25.282913,10.943996,2.252323,0.142140,131.147834,130.308106,6.458789,50.699751,58.255403,21869.306509
min,1.000000,9.430000e+02,0.000000,0.000000,-6.400000,-19.000000,10.806452,42.000000,-13.000000,13.250000,...,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,12.000000,1.001169,0.000000
25%,3.000000,6.237900e+04,1927.000000,40.000000,11.900000,6.000000,29.827586,56.000000,2.000000,31.625000,...,72.000000,6.000000,0.000000,0.000000,1.000000,1.000000,1.000000,88.000000,54.528601,18939.000000
50%,5.000000,9.136700e+04,1951.000000,67.000000,25.000000,11.000000,34.451613,59.000000,9.000000,34.107143,...,84.000000,12.000000,0.000000,0.000000,1.000000,1.000000,1.000000,104.000000,75.293716,37878.000000
75%,6.000000,1.660000e+05,1977.000000,85.000000,42.700000,13.000000,37.322581,62.000000,20.000000,40.879310,...,97.000000,17.000000,0.000000,0.000000,1.000000,1.000000,1.000000,131.000000,97.277534,56817.000000
max,6.000000,6.385382e+06,2015.000000,100.000000,1924.500000,49.000000,64.758065,91.000000,48.000000,65.107143,...,260.000000,185.000000,119.000000,16.000000,360.000000,360.000000,23.300000,311.000000,997.866120,75756.000000


We generate categorical summary statistics for categorical columns.


In [12]:
df_raw.describe(include=['object', 'category'])


,State_Factor,building_class,facility_type
count,75757,75757,75757
unique,7,2,60
top,State_6,Residential,Multifamily_Uncategorized
freq,50840,43558,39455


> **Note on Categorical Data Types:** Depending on the pandas version environment, text columns may be represented using pandas' dedicated `str` data type or legacy `object` data type. In either case, the columns represent standard categorical text features and are processed identically by `OneHotEncoder` and `SimpleImputer(strategy='most_frequent')` without affecting pipeline execution or data leakage guarantees.

We separate numerical and categorical feature names for structured auditing.


In [13]:
raw_num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
raw_cat_cols = df_raw.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Identified {len(raw_num_cols)} Numerical Features: {raw_num_cols[:5]}...")
print(f"Identified {len(raw_cat_cols)} Categorical Features: {raw_cat_cols}")


Identified 61 Numerical Features: ['Year_Factor', 'floor_area', 'year_built', 'energy_star_rating', 'ELEVATION']...
Identified 3 Categorical Features: ['State_Factor', 'building_class', 'facility_type']


## *5. Missing Value Analysis*


We compute absolute missing value counts for columns containing missing entries.


In [14]:
missing_counts = df_raw.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print("Missing Value Counts:\n", missing_counts)


Missing Value Counts:
 days_with_fog                45796
direction_peak_wind_speed    41811
direction_max_wind_speed     41082
max_wind_speed               41082
energy_star_rating           26709
year_built                    1837
dtype: int64


We calculate missing value percentages relative to total dataset size (75,757 rows).


In [15]:
missing_pcts = (df_raw.isnull().mean() * 100).round(2)
missing_pcts = missing_pcts[missing_pcts > 0].sort_values(ascending=False)
print("Missing Value Percentages (%):\n", missing_pcts)


Missing Value Percentages (%):
 days_with_fog                60.45
direction_peak_wind_speed    55.19
direction_max_wind_speed     54.23
max_wind_speed               54.23
energy_star_rating           35.26
year_built                    2.42
dtype: float64


### *Missing Value Interpretation*

The audit reveals missing data concentrated across six specific features:
1. **`days_with_fog` (45,796 missing, 60.45%):** Exceeds the 60% missing threshold due to limited fog monitoring weather stations.
2. **`direction_peak_wind_speed` (41,811 missing, 55.19%), `direction_max_wind_speed` (41,082 missing, 54.23%), `max_wind_speed` (41,082 missing, 54.23%):** High missingness reflecting localized wind instrumentation coverage.
3. **`energy_star_rating` (26,709 missing, 35.26%):** Missing for unrated building types or properties where Energy Star benchmarking was not performed.
4. **`year_built` (1,837 missing, 2.42%):** Minor missingness in municipal property registries.


### *Missing Value Treatment Strategy (Rubric B1)*

- **Drop Strategy (>60% Missing & Identifiers):** Drop `days_with_fog` (60.45% missing) as it lacks sufficient observation density for reliable imputation, alongside `id` (non-predictive primary key).
- **Pipeline Imputation Strategy:** Retain `energy_star_rating` (35.26%) and `year_built` (2.42%), as well as wind metrics. Impute missing numerical values using **median imputation** within the scikit-learn preprocessing pipeline to prevent pre-split data leakage.
- **Missing Indicator Flag:** Engineer `energy_star_missing_flag` to explicitly preserve missingness information as a structural signal for the model.


We execute the dropping of sparse columns (>60% missing) and identifier `id`.


In [16]:
cols_to_drop = ['days_with_fog', 'id']
df_cleaned = df_raw.drop(columns=cols_to_drop, errors='ignore').copy()
df_cleaned['log_site_eui'] = np.log1p(df_cleaned['site_eui'])
print("Dropped sparse/identifier columns:", cols_to_drop)
print("Cleaned Dataset Shape:", df_cleaned.shape)
print("Target Transformation applied: log_site_eui = log1p(site_eui)")


Dropped sparse/identifier columns: ['days_with_fog', 'id']
Cleaned Dataset Shape: (75757, 63)
Target Transformation applied: log_site_eui = log1p(site_eui)


We verify the missing value status after dropping sparse columns.


In [17]:
remaining_missing = df_cleaned.isnull().sum()
print("Remaining Missing Counts per Column:\n", remaining_missing[remaining_missing > 0])


Remaining Missing Counts per Column:
 year_built                    1837
energy_star_rating           26709
direction_max_wind_speed     41082
direction_peak_wind_speed    41811
max_wind_speed               41082
dtype: int64


## *6. Duplicate Analysis*


We check for completely identical duplicate rows across all attributes in an explicit check cell (Rubric B1).


In [18]:
dup_rows = df_raw.duplicated().sum()
print("Total Duplicate Rows in Raw Dataset:", dup_rows)


Total Duplicate Rows in Raw Dataset: 0


We check for duplicate primary keys in the `id` column.


In [19]:
if 'id' in df_raw.columns:
    dup_ids = df_raw['id'].duplicated().sum()
    print("Total Duplicate IDs in Raw Dataset:", dup_ids)
else:
    print("No 'id' column found.")


Total Duplicate IDs in Raw Dataset: 0


### Duplicate Treatment Decision (Rubric B1)

The duplicate audit identified **0 completely duplicated rows** and **0 duplicated IDs** in the raw dataset. Therefore, no duplicate records required removal. The dataset proceeds to the subsequent cleaning and feature-engineering steps without duplicate-row treatment.